In [1]:
import sys
print(sys.executable)

D:\Anaconda_file\envs\retailpulse\python.exe


In [2]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

# Load RFM customer data
rfm = pd.read_csv("../data/rfm_customer_segments.csv")

print("RFM dataset loaded successfully!")
print("Shape:", rfm.shape)
print("Columns:", rfm.columns.tolist())

RFM dataset loaded successfully!
Shape: (1200, 9)
Columns: ['Customer ID', 'Recency', 'Frequency', 'Monetary', 'R_Score', 'F_Score', 'M_Score', 'RFM_Score', 'Segment']


In [3]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

# Select RFM features
rfm_features = rfm[["Recency", "Frequency", "Monetary"]]

# Scale the features
scaler = StandardScaler()
rfm_scaled = scaler.fit_transform(rfm_features)

print("RFM features scaled successfully!")
print("Scaled data shape:", rfm_scaled.shape)

RFM features scaled successfully!
Scaled data shape: (1200, 3)


In [4]:
import matplotlib.pyplot as plt

inertia = []

for k in range(2, 9):
    kmeans = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )
    kmeans.fit(rfm_scaled)
    inertia.append(kmeans.inertia_)

print("K values tested:", list(range(2, 9)))
print("Inertia values:", inertia)

plt.figure(figsize=(8, 5))
plt.plot(range(2, 9), inertia, marker="o")
plt.xlabel("Number of Clusters (K)")
plt.ylabel("Inertia")
plt.title("Elbow Method for Optimal K")
plt.grid(True)
plt.show()

K values tested: [2, 3, 4, 5, 6, 7, 8]
Inertia values: [1300.6167779799632, 743.7461990834852, 516.9362357914407, 387.12713813670695, 317.9715581043908, 269.70261022373865, 232.0401566981894]


<Figure size 800x500 with 1 Axes>

In [5]:
# Final K-Means model
kmeans = KMeans(
    n_clusters=4,
    random_state=42,
    n_init=10
)

rfm["Cluster"] = kmeans.fit_predict(rfm_scaled)

print("K-Means clustering completed!")
print("\nCluster distribution:")
print(rfm["Cluster"].value_counts().sort_index())

K-Means clustering completed!

Cluster distribution:
Cluster
0    418
1    418
2     81
3    283
Name: count, dtype: int64


In [6]:
cluster_profile = rfm.groupby("Cluster").agg(
    Customers=("Customer ID", "count"),
    Avg_Recency=("Recency", "mean"),
    Avg_Frequency=("Frequency", "mean"),
    Avg_Monetary=("Monetary", "mean")
).round(2)

print("Cluster profiles:")
display(cluster_profile)

Cluster profiles:


,Customers,Avg_Recency,Avg_Frequency,Avg_Monetary
Cluster,,,,
0,418,509.70,21.40,645.97
1,418,411.56,34.64,1424.00
2,81,186.33,65.73,4629.25
3,283,308.34,49.18,2666.90


In [7]:
cluster_names = {
    0: "Inactive Customers",
    1: "Occasional Customers",
    2: "VIP Customers",
    3: "Loyal Customers"
}

rfm["Cluster_Name"] = rfm["Cluster"].map(cluster_names)

print("Business cluster names assigned!")

print("\nCluster distribution:")
print(rfm["Cluster_Name"].value_counts())

display(
    rfm[
        ["Customer ID", "Recency", "Frequency", "Monetary",
         "Cluster", "Cluster_Name"]
    ].head(10)
)

Business cluster names assigned!

Cluster distribution:
Cluster_Name
Occasional Customers    418
Inactive Customers      418
Loyal Customers         283
VIP Customers            81
Name: count, dtype: int64


,Customer ID,Recency,Frequency,Monetary,Cluster,Cluster_Name
0,10000.0,158,69,5658.65,2,VIP Customers
1,10001.0,225,45,2675.70,3,Loyal Customers
2,10002.0,397,42,1700.28,1,Occasional Customers
3,10003.0,310,45,2157.64,3,Loyal Customers
4,10004.0,349,40,2421.04,3,Loyal Customers
5,10005.0,470,22,658.57,0,Inactive Customers
6,10006.0,264,51,3646.31,3,Loyal Customers
7,10007.0,462,26,786.79,0,Inactive Customers
8,10008.0,442,37,1420.14,1,Occasional Customers
9,10009.0,372,49,2162.16,3,Loyal Customers


In [8]:
segmentation_path = "../data/customer_segmentation.csv"

rfm.to_csv(segmentation_path, index=False)

print("Customer segmentation saved successfully!")
print("File:", segmentation_path)
print("Customers:", len(rfm))

Customer segmentation saved successfully!
File: ../data/customer_segmentation.csv
Customers: 1200
